# 01 — Introduction to CUDA & GPU Programming

**What you'll learn:**
- What CUDA is and how GPUs differ from CPUs
- How to verify your GPU setup in Python
- Basic CUDA concepts: threads, blocks, grids
- Your first GPU computation using PyTorch

## 1. CPU vs GPU — The Core Difference

| | CPU | GPU (RTX 3050) |
|---|---|---|
| Cores | ~8–16 big cores | 2048 CUDA cores |
| Strength | Complex sequential tasks | Massive parallel tasks |
| Memory | ~16–64 GB RAM | 4 GB VRAM |
| Best for | Logic, branching, I/O | Matrix ops, deep learning |

**Key idea:** GPUs run thousands of threads simultaneously. CUDA is NVIDIA's platform to program them.

## 2. CUDA Hierarchy

```
Grid  (entire GPU job)
 └── Blocks  (group of threads, share fast memory)
      └── Threads  (individual worker)
```

- A **thread** does one unit of work
- A **block** groups threads that can cooperate via shared memory
- A **grid** is the collection of all blocks for a kernel launch

In [1]:
# Step 1: Verify GPU is available
import torch

print(f"PyTorch version    : {torch.__version__}")
print(f"CUDA available     : {torch.cuda.is_available()}")
print(f"CUDA version       : {torch.version.cuda}")
print(f"GPU name           : {torch.cuda.get_device_name(0)}")
print(f"Total VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"CUDA cores (SM)    : {torch.cuda.get_device_properties(0).multi_processor_count} SMs")

PyTorch version    : 2.11.0+cu128
CUDA available     : True
CUDA version       : 12.8
GPU name           : NVIDIA GeForce RTX 3050 Laptop GPU
Total VRAM         : 3.95 GB
CUDA cores (SM)    : 16 SMs


## What is a Tensor?

A **tensor** is a multi-dimensional array — a generalization of scalars, vectors, and matrices to arbitrary dimensions.

| Dimensions | Name | Example |
|---|---|---|
| 0-D | Scalar | `42` |
| 1-D | Vector | `[1, 2, 3]` |
| 2-D | Matrix | `[[1,2],[3,4]]` |
| 3-D+ | Tensor | `shape (batch, height, width)` |

In PyTorch:
- Tensors are the core data structure — inputs, weights, and gradients are all tensors
- They live on a **device**: CPU RAM or GPU VRAM
- They have a **dtype**: `float32`, `float16`, `int64`, etc.
- They support automatic differentiation via `requires_grad=True`

The key insight: a tensor moved to GPU lives in **VRAM** and all operations on it run on CUDA cores — that's what makes GPU-accelerated deep learning fast.

In [2]:
# Step 2: Move data to GPU
device = torch.device("cuda")  # use GPU

# Create tensor on CPU
cpu_tensor = torch.tensor([1.0, 2.0, 3.0, 4.0])
print(f"CPU tensor device: {cpu_tensor.device}")

# Move to GPU
gpu_tensor = cpu_tensor.to(device)
print(f"GPU tensor device: {gpu_tensor.device}")

CPU tensor device: cpu
GPU tensor device: cuda:0


In [5]:
# Step 3: GPU vs CPU speed comparison
import time

size = 10000
A = torch.randn(size, size)
B = torch.randn(size, size)

# CPU
start = time.time()
C_cpu = A @ B
cpu_time = time.time() - start
print(f"CPU matrix multiply ({size}x{size}): {cpu_time:.3f}s")

# GPU
A_gpu = A.to(device)
B_gpu = B.to(device)
torch.cuda.synchronize()  # wait for GPU to be ready

start = time.time()
C_gpu = A_gpu @ B_gpu
torch.cuda.synchronize()  # wait for GPU to finish
gpu_time = time.time() - start
print(f"GPU matrix multiply ({size}x{size}): {gpu_time:.3f}s")
print(f"Speedup: {cpu_time/gpu_time:.1f}x")

CPU matrix multiply (10000x10000): 3.687s
GPU matrix multiply (10000x10000): 0.530s
Speedup: 7.0x


In [6]:
# Step 4: Check memory usage
print(f"Allocated VRAM : {torch.cuda.memory_allocated() / 1e6:.1f} MB")
print(f"Reserved VRAM  : {torch.cuda.memory_reserved() / 1e6:.1f} MB")

# Free GPU memory
del A_gpu, B_gpu, C_gpu
torch.cuda.empty_cache()
print(f"After cleanup  : {torch.cuda.memory_allocated() / 1e6:.1f} MB")

Allocated VRAM : 1209.1 MB
Reserved VRAM  : 3602.9 MB
After cleanup  : 8.5 MB


## Summary
- Use `torch.cuda.is_available()` to check GPU availability
- Use `.to('cuda')` or `.cuda()` to move tensors to GPU
- Always call `torch.cuda.synchronize()` before timing GPU ops
- GPUs shine on large parallel workloads like matrix multiplication